Data Scraping For Monetary Policy Project --> As per Suggestion on Document I have been given the United Kingdom - Inflation CPI - Funds rate - Frequency = Quarterly

## Packages

In [12]:
import pandas as pd
import requests
import numpy as np

### Prices, CPI Growth

In [13]:
# Getting Data from Fred

API_KEY = "51286e2abae242cda74d0d166f50b652"
SERIES_ID_CPI_Growth = "CPALTT01GBQ657N"

url = "https://api.stlouisfed.org/fred/series/observations"

parameters = {
    "series_id": SERIES_ID_CPI_Growth,
    "api_key": API_KEY,
    "file_type": "json",
    "observation_start": "1955-04-01",
    "observation_end": "2023-10-01"
}
response = requests.get(url, params=parameters)
response.raise_for_status()
data_fred_CPI_Growth = response.json()
# making the observations into a dataframe
quarterly_cpi_growth_unitedkingdom_fred_data_raw_df = pd.DataFrame(data_fred_CPI_Growth["observations"])
quarterly_cpi_growth_unitedkingdom_fred_data_df = quarterly_cpi_growth_unitedkingdom_fred_data_raw_df[["date", "value"]].copy()
quarterly_cpi_growth_unitedkingdom_fred_data_df["date"] = pd.to_datetime(quarterly_cpi_growth_unitedkingdom_fred_data_df["date"])
quarterly_cpi_growth_unitedkingdom_fred_data_df["value"] = pd.to_numeric(quarterly_cpi_growth_unitedkingdom_fred_data_df["value"], errors="coerce")
quarterly_cpi_growth_unitedkingdom_fred_data_df = quarterly_cpi_growth_unitedkingdom_fred_data_df.set_index("date")

# Renaming the columns in my data frame so that they remain consistent when I have to go on and merge them 
quarterly_cpi_growth_unitedkingdom_fred_data_df.index = pd.PeriodIndex(quarterly_cpi_growth_unitedkingdom_fred_data_df.index, freq="Q")
quarterly_cpi_growth_unitedkingdom_fred_data_df = quarterly_cpi_growth_unitedkingdom_fred_data_df.rename(columns={"value": "CPI Growth Rate"})

# transfomring it into log levels to mimic Inflation growth as per the slides
quarterly_cpi_growth_unitedkingdom_fred_data_df.head()



,CPI Growth Rate
date,
1955Q2,1.119177
1955Q3,1.405923
1955Q4,2.300885
1956Q1,0.607093
1956Q2,2.090939


CPI 

In [14]:
API_KEY = "51286e2abae242cda74d0d166f50b652"
SERIES_ID_CPI_Growth = "CPIUKQ"
url = "https://api.stlouisfed.org/fred/series/observations"
parameters = {
    "series_id": SERIES_ID_CPI_Growth,
    "api_key": API_KEY,
    "file_type": "json",
    "observation_start": "1955-04-01",
    "observation_end": "2023-10-01"
}
response = requests.get(url, params=parameters)
response.raise_for_status()
data_fred_CPI = response.json()

# Fix: extract "observations" from the dict first
quarterly_cpi_unitedkingdom_fred_data_df = pd.DataFrame(data_fred_CPI["observations"])[["date", "value"]].copy()
quarterly_cpi_unitedkingdom_fred_data_df["date"]  = pd.to_datetime(quarterly_cpi_unitedkingdom_fred_data_df["date"])
quarterly_cpi_unitedkingdom_fred_data_df["value"] = pd.to_numeric(quarterly_cpi_unitedkingdom_fred_data_df["value"], errors="coerce")
quarterly_cpi_unitedkingdom_fred_data_df = quarterly_cpi_unitedkingdom_fred_data_df.set_index("date")
quarterly_cpi_unitedkingdom_fred_data_df.index = pd.PeriodIndex(quarterly_cpi_unitedkingdom_fred_data_df.index, freq="Q")
quarterly_cpi_unitedkingdom_fred_data_df = quarterly_cpi_unitedkingdom_fred_data_df.rename(columns={"value": "CPI"})

# Log first difference (quarter on quarter)
quarterly_cpi_unitedkingdom_fred_data_df["Log_Dif_Growth"] = np.log(quarterly_cpi_unitedkingdom_fred_data_df["CPI"]).diff(1)

# Log year-on-year growth (annualised)
quarterly_cpi_unitedkingdom_fred_data_df["Log_YoY_Growth"] = np.log(quarterly_cpi_unitedkingdom_fred_data_df["CPI"]).diff(4) * 100

quarterly_cpi_unitedkingdom_fred_data_df.head(10)

,CPI,Log_Dif_Growth,Log_YoY_Growth
date,,,
1955Q2,5.73,NaN,NaN
1955Q3,5.81,0.013865,NaN
1955Q4,5.99,0.030511,NaN
1956Q1,6.02,0.004996,NaN
1956Q2,6.12,0.016475,6.584657
1956Q3,6.10,-0.003273,4.870820
1956Q4,6.17,0.011410,2.960743
1957Q1,6.25,0.012883,3.749420
1957Q2,6.28,0.004789,2.580788


GDP Deflator

In [15]:
# GDP Deflator - Fred Data Base 

SERIES_ID_GDP_DEFLATE = "NGDPDSAIXGBQ"

parameters = {
    "series_id": SERIES_ID_GDP_DEFLATE,
    "api_key": API_KEY,
    "file_type": "json",
    "observation_start": "1955-04-01",
    "observation_end": "2026-01-01"
}

response = requests.get(url, params=parameters)
response.raise_for_status()
data_fred_GDP_Deflator = response.json()


quarterly_gdp_deflator_data_raw_df = pd.DataFrame(data_fred_GDP_Deflator["observations"])
quarterly_gdp_deflator_data_df = quarterly_gdp_deflator_data_raw_df[["date","value"]]
quarterly_gdp_deflator_data_df["date"] = pd.to_datetime(quarterly_gdp_deflator_data_df["date"])
quarterly_gdp_deflator_data_df["value"] = pd.to_numeric(quarterly_gdp_deflator_data_df["value"], errors="coerce")
quarterly_gdp_deflator_data_df = quarterly_gdp_deflator_data_df.set_index("date")
quarterly_gdp_deflator_data_df.index = pd.PeriodIndex(quarterly_gdp_deflator_data_df.index, freq="Q")

# Transforming into log levels to mimic inflation

quarterly_gdp_deflator_data_df["GDP Deflator_logged"] = np.log(quarterly_gdp_deflator_data_df["value"])
quarterly_gdp_deflator_data_df["GDP Deflators CPI Proxy"] = quarterly_gdp_deflator_data_df["GDP Deflator_logged"].diff()

quarterly_gdp_deflator_data_df = quarterly_gdp_deflator_data_df[["GDP Deflator_logged", "GDP Deflators CPI Proxy"]].dropna()

quarterly_gdp_deflator_data_df.head()


/var/folders/k1/mhyzg0w15gb04d0x6q7gdp2w0000gn/T/ipykernel_41599/269056796.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  quarterly_gdp_deflator_data_df["date"] = pd.to_datetime(quarterly_gdp_deflator_data_df["date"])
/var/folders/k1/mhyzg0w15gb04d0x6q7gdp2w0000gn/T/ipykernel_41599/269056796.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  quarterly_gdp_deflator_data_df["value"] = pd.to_numeric(quarterly_gdp_deflator_data_df["value"], errors="coerce")


,GDP Deflator_logged,GDP Deflators CPI Proxy
date,,
1955Q3,1.520057,0.016863
1955Q4,1.544279,0.024221
1956Q1,1.559656,0.015377
1956Q2,1.574166,0.014510
1956Q3,1.590796,0.016631


Real activity and output

Real GDP 

In [16]:
# Real GDP - Fred Data Base 

SERIES_ID_REAL_GDP = "NGDPRSAXDCGBQ" 

parameters = {
    "series_id": SERIES_ID_REAL_GDP,
    "api_key": API_KEY,
    "file_type": "json",
    "observation_start": "1955-04-01",
    "observation_end": "2025-01-01"
}

response = requests.get(url, params=parameters)
response.raise_for_status()
data_fred_real_gdp = response.json()

quarterly_uk_real_gdp_df = pd.DataFrame(data_fred_real_gdp["observations"])[["date", "value"]]
quarterly_uk_real_gdp_df["date"] = pd.to_datetime(quarterly_uk_real_gdp_df["date"])
quarterly_uk_real_gdp_df["value"] = pd.to_numeric(quarterly_uk_real_gdp_df["value"], errors="coerce")
# Rename
quarterly_uk_real_gdp_df = quarterly_uk_real_gdp_df.rename(columns={"value": "RealGDP"})

# setting the date as the index 
quarterly_uk_real_gdp_df = quarterly_uk_real_gdp_df.set_index("date")
quarterly_uk_real_gdp_df.index = pd.PeriodIndex(quarterly_uk_real_gdp_df.index, freq='Q')

quarterly_uk_real_gdp_df["RealGDP_Log"] = np.log(quarterly_uk_real_gdp_df["RealGDP"])
quarterly_uk_real_gdp_df["RealGDP_Log_Change"] = quarterly_uk_real_gdp_df["RealGDP_Log"].diff()

quarterly_uk_real_gdp_df.head()


,RealGDP,RealGDP_Log,RealGDP_Log_Change
date,,,
1955Q2,145551.0,11.888282,NaN
1955Q3,147995.0,11.904934,0.016652
1955Q4,147119.0,11.898997,-0.005937
1956Q1,148955.0,11.911400,0.012402
1956Q2,148835.0,11.910594,-0.000806


Policy Rate

In [17]:
SERIES_ID_POLICY_RATE = "BOERUKM" 
parameters = {
    "series_id": SERIES_ID_POLICY_RATE,
    "api_key": API_KEY,
    "file_type": "json",
    "observation_start": "1955-04-01",
    "observation_end": "2025-01-01"
}
response = requests.get(url, params=parameters)
response.raise_for_status()
data_fred_policy_rate = response.json()

monthly_uk_policy_rate_df = pd.DataFrame(data_fred_policy_rate["observations"])[["date", "value"]].copy()
monthly_uk_policy_rate_df["date"] = pd.to_datetime(monthly_uk_policy_rate_df["date"])
monthly_uk_policy_rate_df["value"] = pd.to_numeric(monthly_uk_policy_rate_df["value"], errors="coerce")
monthly_uk_policy_rate_df = monthly_uk_policy_rate_df.rename(columns={"value": "PolicyRate"})
monthly_uk_policy_rate_df = monthly_uk_policy_rate_df.set_index("date")
quarterly_uk_policy_rate_df = monthly_uk_policy_rate_df.resample("Q").mean()
quarterly_uk_policy_rate_df.index = quarterly_uk_policy_rate_df.index.to_period("Q") 
quarterly_uk_policy_rate_df["PolicyRate_Change"] = quarterly_uk_policy_rate_df["PolicyRate"].diff()

quarterly_uk_policy_rate_df.head()

/var/folders/k1/mhyzg0w15gb04d0x6q7gdp2w0000gn/T/ipykernel_41599/1350070602.py:18: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  quarterly_uk_policy_rate_df = monthly_uk_policy_rate_df.resample("Q").mean()


,PolicyRate,PolicyRate_Change
date,,
1955Q2,4.500000,NaN
1955Q3,4.500000,0.000000
1955Q4,4.500000,0.000000
1956Q1,5.166667,0.666667
1956Q2,5.500000,0.333333


3-month interest rate, United Kingdom & Growth Rate 

In [18]:
SERIES_ID_SHORT = "IR3TTS01GBM156N"
params = {
    "series_id": SERIES_ID_SHORT,
    "api_key": API_KEY,
    "file_type": "json",
    "observation_start": "1955-04-01",
    "observation_end": "2025-01-01"
}
response = requests.get(url, params=params)
response.raise_for_status()
data_fred_short = response.json()

monthly_uk_short_rate_df = pd.DataFrame(data_fred_short["observations"])[["date", "value"]].copy()
monthly_uk_short_rate_df["date"] = pd.to_datetime(monthly_uk_short_rate_df["date"])
monthly_uk_short_rate_df["value"] = pd.to_numeric(monthly_uk_short_rate_df["value"], errors="coerce")
monthly_uk_short_rate_df = monthly_uk_short_rate_df.rename(columns={"value": "ShortRate"})
monthly_uk_short_rate_df = monthly_uk_short_rate_df.set_index("date")

quarterly_uk_short_rate_df = monthly_uk_short_rate_df.resample("Q").mean()
quarterly_uk_short_rate_df.index = quarterly_uk_short_rate_df.index.to_period("Q") 
quarterly_uk_short_rate_df["ShortRate_Change"] = quarterly_uk_short_rate_df["ShortRate"].diff()

quarterly_uk_short_rate_df.head()

/var/folders/k1/mhyzg0w15gb04d0x6q7gdp2w0000gn/T/ipykernel_41599/580450449.py:19: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  quarterly_uk_short_rate_df = monthly_uk_short_rate_df.resample("Q").mean()


,ShortRate,ShortRate_Change
date,,
1955Q2,NaN,NaN
1955Q3,NaN,NaN
1955Q4,NaN,NaN
1956Q1,NaN,NaN
1956Q2,NaN,NaN


SCALING VARIABLES

Merging All the dataframes into one big final data frame

In [19]:
final_df = (
    quarterly_uk_short_rate_df.join(quarterly_uk_policy_rate_df, how="inner")
    .join(quarterly_uk_real_gdp_df, how='inner')
    .join(quarterly_gdp_deflator_data_df, how='inner')
    .join(quarterly_cpi_unitedkingdom_fred_data_df, how = 'inner')
    .join(quarterly_cpi_growth_unitedkingdom_fred_data_df, how ='inner')
    .loc["1960Q2": "2017Q1"]
)

final_df

,ShortRate,ShortRate_Change,PolicyRate,PolicyRate_Change,RealGDP,RealGDP_Log,RealGDP_Log_Change,GDP Deflator_logged,GDP Deflators CPI Proxy,CPI,Log_Dif_Growth,Log_YoY_Growth,CPI Growth Rate
date,,,,,,,,,,,,,
1960Q2,4.963333,0.393333,5.333333,0.333333,168948.0,12.037346,-0.006919,1.677209,0.016328,6.59,0.003040,1.221389,0.606980
1960Q3,5.570000,0.606667,6.000000,0.666667,171442.0,12.052000,0.014654,1.676828,-0.000382,6.59,0.000000,1.221389,0.150830
1960Q4,4.693333,-0.876667,5.333333,-0.666667,172382.0,12.057468,0.005468,1.691078,0.014250,6.70,0.016554,1.655418,1.054217
1961Q1,4.353333,-0.340000,5.000000,-0.333333,175276.0,12.074117,0.016649,1.699609,0.008531,6.72,0.002981,2.257432,0.536513
1961Q2,4.463333,0.110000,5.000000,0.000000,175974.0,12.078092,0.003974,1.695512,-0.004097,6.77,0.007413,2.694774,1.245182
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015Q4,0.471600,0.003033,0.500000,0.000000,621968.0,13.340644,0.005975,4.687885,0.005442,100.29,0.001497,0.059844,0.199601
2016Q1,0.469233,-0.002367,0.500000,0.000000,625099.0,13.345665,0.005021,4.693441,0.005555,99.84,-0.004497,0.351177,-0.298805
2016Q2,0.423133,-0.046100,0.500000,0.000000,629697.0,13.352994,0.007329,4.696005,0.002564,100.43,0.005892,0.359103,0.699301


In [20]:
final_df.to_csv("final_df.csv")